In [2]:
# !pip install faiss-cpu
# !pip install python-dotenv
# !pip install openai

In [3]:
import numpy as np
import json
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI

In [4]:
#Load data + FAISS
embeddings = np.load("../processed/embeddings.npy")

with open("../processed/chunks.json") as f:
    all_chunks = json.load(f)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [5]:
#Load embedding model (for query + grounding)
model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
#Search function
def search(query, k=3):
    # convert query → embedding
    query_embedding = model.encode([query])

    # search FAISS
    distances, indices = index.search(query_embedding, k)

    results = []
    for idx in indices[0]:
        results.append(all_chunks[idx])

    return results

In [7]:
results = search("What are GDPR data protection rules?")

for r in results:
    print("\n---")
    print(r["text"])


---
Clause (92): The protection of the rights and freedoms of natural persons with regard to the processing of personal data require that appropriate technical and organisational measures be taken to ensure that the requirements of this Regulation are met. In order to be able to demonstrate compliance with this Regulation, the controller should adopt internal policies and implement measures which meet in particular the principles of data protection by design and data protection by default. Such measures could consist, inter alia, of minimising the processing of personal data, pseudonymising personal data as soon as possible, transparency with regard to the functions and processing of personal data, enabling the data subject to monitor the data processing, enabling the controller to create and improve security features. When developing, designing, selecting and using applications, services and products that are based on the processing of personal data or process personal data to fulfil

In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

In [9]:
print(api_key[:10])

sk-or-v1-e


In [11]:
# Setup OpenRouter
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

In [12]:
def generate_answer(query, context):
    prompt = f"""
You are a compliance assistant.

Answer ONLY using the context below.
Include citations like (Clause X) when referring to specific rules.

If the answer is not in the context, say: "Not found in provided documents."

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b:free", 
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [13]:
def filter_sources(results, query):
    filtered = []
    for r in results:
        if any(word.lower() in r["text"].lower() for word in query.split()):
            filtered.append(r["source"])
    return list(set(filtered))

In [14]:

def grounding_score(answer, context):
    ans_emb = model.encode([answer])
    ctx_emb = model .encode([context])
    return np.dot(ans_emb, ctx_emb.T)[0][0]

def check_hallucination(answer, context, threshold=0.5):
    score = grounding_score(answer, context)

    if score < threshold:
        return "⚠️ Potential hallucination", score
    else:
        return "✅ Grounded", score

In [15]:
#semantic
def filter_relevant_chunks(results, query):
    query_emb = model.encode([query])

    filtered = []
    for r in results:
        chunk_emb = model.encode([r["text"]])
        score = np.dot(query_emb, chunk_emb.T)[0][0]

        if score > 0.5:  # tune this
            filtered.append(r)

    return filtered if filtered else results[:3]

In [16]:
def ask(query):
    results = search(query, k=3)

    results = filter_relevant_chunks(results, query)

    context = "\n\n".join([r["text"] for r in results])

    answer = generate_answer(query, context)

    status, score = check_hallucination(answer, context)

    return {
        "answer": answer,
        "status": status,
        "score": float(score),
        "sources": list(set(
    r["source"] for r in results if "Clause" in r["text"]
)),
        "evidence": [
            {
                "source": r["source"],
                "snippet": r["text"][:150] + "..."
            }
            for r in results
        ]
    }

In [17]:
response = ask("What safeguards are required for transferring personal data under GDPR?")

In [18]:
print(response)

{'answer': 'The GDPR requires that any transfer of personal data to a third country or an international organisation be protected by **appropriate safeguards** and that enforceable data‑subject rights and effective legal remedies remain available (Clause\u202f294).  \n\nThese safeguards can be provided without a specific supervisory‑authority authorisation by using one of the following mechanisms (Clause\u202f294\u202f(2)):\n\n1. **A legally binding and enforceable instrument** between public authorities or bodies.  \n2. **Binding corporate rules** (BCRs) in accordance with Article\u202f47.  \n3. **Standard data‑protection clauses** adopted by the European Commission (the “standard contractual clauses”).  \n\nIn all cases the safeguards must ensure that the level of protection guaranteed by the GDPR is not undermined (Clause\u202f115).  \n\n**Therefore, the required safeguards are:** appropriate safeguards (including enforceable rights), implemented via a legally binding instrument, bi

In [19]:
import faiss

faiss.write_index(index, "faiss.index")